# NBA Broadcast Slot Scheduling with CP-SAT

This notebook describes and implements a Constraint Satisfaction Problem (CSP) for assigning NBA games to TV broadcast slots using `ortools.sat.python.cp_model`.

The model is designed from an operations research perspective: it separates feasibility constraints, business constraints, objective design, solving, and result extraction. That separation makes the formulation easier to audit and easier to evolve when a future iteration needs to convert a hard rule into a soft rule with penalties.

## 1. Setup & Environment

We use `pandas` for tabular input/output inspection and OR-Tools CP-SAT for the binary optimization model.

The core solver import is:

```python
from ortools.sat.python import cp_model
```

CP-SAT is appropriate here because each game-slot assignment is naturally represented as a binary decision variable.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import Dict, List, Sequence, Tuple

import pandas as pd
from ortools.sat.python import cp_model

@dataclass(frozen=True)
class Team:
    """NBA team metadata used by the scheduling model."""

    team_id: str
    name: str
    conference: str


@dataclass(frozen=True)
class Game:
    """Game demand that must be assigned to exactly one broadcast slot."""

    game_id: str
    home_team: str
    away_team: str
    is_big_match: bool


@dataclass(frozen=True)
class Slot:
    """Available TV inventory for a specific channel, day, and broadcast window."""

    slot_id: str
    day: str
    start_time: str
    channel: str
    is_prime_time: bool


AssignmentVars = Dict[Tuple[str, str], cp_model.IntVar]

## 2. Input Data Mockup

The model consumes three data structures:

- `Teams`: team metadata used for team-level conflict constraints.
- `Games`: game demand, including `is_big_match` to identify marquee games.
- `Slots`: TV inventory, including `is_prime_time`, `channel`, `day`, and `start_time`.

The extra `start_time` field supports the business rule that no two big matches should air on the same channel simultaneously.

In [ ]:
def create_mock_data() -> Tuple[List[Team], List[Game], List[Slot]]:
    """Create realistic, compact input data for model demonstration.

    Returns:
        A tuple containing teams, games, and available TV slots.
    """

    teams = [
        Team("BOS", "Boston Celtics", "East"),
        Team("NYK", "New York Knicks", "East"),
        Team("CHI", "Chicago Bulls", "East"),
        Team("PHI", "Philadelphia 76ers", "East"),
        Team("LAL", "Los Angeles Lakers", "West"),
        Team("GSW", "Golden State Warriors", "West"),
        Team("DEN", "Denver Nuggets", "West"),
        Team("MIA", "Miami Heat", "East"),
        Team("DAL", "Dallas Mavericks", "West"),
        Team("PHX", "Phoenix Suns", "West"),
        Team("MEM", "Memphis Grizzlies", "West"),
        Team("SAS", "San Antonio Spurs", "West"),
    ]

    games = [
        Game("G1", "BOS", "NYK", True),
        Game("G2", "LAL", "GSW", True),
        Game("G3", "DEN", "MIA", True),
        Game("G4", "DAL", "PHX", False),
        Game("G5", "CHI", "PHI", False),
        Game("G6", "MEM", "SAS", False),
        Game("G7", "BOS", "LAL", True),
        Game("G8", "GSW", "DEN", True),
        Game("G9", "MIA", "DAL", False),
        Game("G10", "PHX", "MEM", False),
        Game("G11", "PHI", "CHI", False),
        Game("G12", "SAS", "NYK", False),
    ]

    slots = [
        Slot("S1", "Monday", "19:00", "ESPN", False),
        Slot("S2", "Monday", "21:30", "TNT", True),
        Slot("S3", "Tuesday", "19:00", "ESPN", False),
        Slot("S4", "Tuesday", "21:30", "TNT", True),
        Slot("S5", "Wednesday", "19:00", "ESPN", False),
        Slot("S6", "Wednesday", "21:30", "TNT", True),
        Slot("S7", "Thursday", "19:00", "NBA TV", False),
        Slot("S8", "Thursday", "21:30", "ESPN", True),
        Slot("S9", "Friday", "19:00", "NBA TV", False),
        Slot("S10", "Friday", "21:30", "ESPN", True),
        Slot("S11", "Saturday", "19:00", "TNT", True),
        Slot("S12", "Saturday", "21:30", "ABC", True),
        Slot("S13", "Sunday", "19:00", "TNT", True),
        Slot("S14", "Sunday", "21:30", "ABC", True),
    ]

    return teams, games, slots


teams, games, slots = create_mock_data()

pd.DataFrame([asdict(game) for game in games])

In [ ]:
pd.DataFrame([asdict(slot) for slot in slots])

## 3. Model Initialization

Let \(M\) be the set of games and \(S\) be the set of broadcast slots.

The binary decision variable is:

$$
x_{m,s} =
\begin{cases}
1, & \text{if game } m \text{ is assigned to slot } s \\
0, & \text{otherwise}
\end{cases}
$$

for every \(m \in M\) and \(s \in S\).

In [ ]:
def initialize_model(
    games: Sequence[Game], slots: Sequence[Slot]
) -> Tuple[cp_model.CpModel, AssignmentVars]:
    model = cp_model.CpModel()
    x = {
        (game.game_id, slot.slot_id): model.NewBoolVar(
            f"x_{game.game_id}_{slot.slot_id}"
        )
        for game in games
        for slot in slots
    }
    return model, x

## 4. Hard Constraints

### Assignment Constraint

Each game must be assigned to exactly one broadcast slot:

$$
\sum_{s \in S} x_{m,s} = 1 \quad \forall m \in M
$$

### Slot Resource Constraint

Each slot can host at most one game:

$$
\sum_{m \in M} x_{m,s} \leq 1 \quad \forall s \in S
$$

### Team Conflict Constraint

A team cannot play more than one game on the same day. Let \(M_t\) be the games involving team \(t\), and \(S_d\) be the slots on day \(d\):

$$
\sum_{m \in M_t} \sum_{s \in S_d} x_{m,s} \leq 1 \quad \forall t \in T, d \in D
$$

In [ ]:
def add_hard_constraints(
    model: cp_model.CpModel,
    x: AssignmentVars,
    games: Sequence[Game],
    slots: Sequence[Slot],
    teams: Sequence[Team],
) -> None:
    # Complete schedule: every game appears exactly once.
    for game in games:
        model.Add(sum(x[(game.game_id, slot.slot_id)] for slot in slots) == 1)

    # Broadcast inventory: each slot has capacity for one game.
    for slot in slots:
        model.Add(sum(x[(game.game_id, slot.slot_id)] for game in games) <= 1)

    # Player/team welfare and operational realism: no team plays twice per day.
    days = sorted({slot.day for slot in slots})
    for team in teams:
        team_games = [
            game for game in games if team.team_id in {game.home_team, game.away_team}
        ]
        for day in days:
            day_slots = [slot for slot in slots if slot.day == day]
            model.Add(
                sum(
                    x[(game.game_id, slot.slot_id)]
                    for game in team_games
                    for slot in day_slots
                )
                <= 1
            )

## 5. Soft / Business Constraints

These rules are implemented as hard constraints in this first version, but they are isolated in `add_business_constraints()` so they can later be relaxed with penalty variables.

### Prime-Time Capacity

Let \(S^P\) be the set of prime-time slots and \(N\) be the maximum number of games allowed in prime-time inventory:

$$
\sum_{m \in M} \sum_{s \in S^P} x_{m,s} \leq N
$$

### Big Match Distribution

Let \(M^B\) be the set of big matches. For each broadcast window \((d, c, h)\), where \(d\) is day, \(c\) is channel, and \(h\) is start time, no more than one big match can be scheduled simultaneously on that channel:

$$
\sum_{m \in M^B} \sum_{s \in S_{d,c,h}} x_{m,s} \leq 1
\quad \forall (d,c,h)
$$

In [ ]:
def add_business_constraints(
    model: cp_model.CpModel,
    x: AssignmentVars,
    games: Sequence[Game],
    slots: Sequence[Slot],
    max_prime_time_games: int,
) -> None:
    prime_time_slots = [slot for slot in slots if slot.is_prime_time]
    model.Add(
        sum(
            x[(game.game_id, slot.slot_id)]
            for game in games
            for slot in prime_time_slots
        )
        <= max_prime_time_games
    )

    big_games = [game for game in games if game.is_big_match]
    broadcast_windows = sorted(
        {(slot.day, slot.channel, slot.start_time) for slot in slots}
    )
    for day, channel, start_time in broadcast_windows:
        matching_slots = [
            slot
            for slot in slots
            if slot.day == day and slot.channel == channel and slot.start_time == start_time
        ]
        model.Add(
            sum(
                x[(game.game_id, slot.slot_id)]
                for game in big_games
                for slot in matching_slots
            )
            <= 1
        )

## 6. Objective Function

The business objective is to maximize marquee-game exposure by assigning big matches to prime-time slots.

Let \(M^B\) be big matches and \(S^P\) be prime-time slots:

$$
\max \sum_{m \in M^B} \sum_{s \in S^P} x_{m,s}
$$

This objective is intentionally simple and interpretable. Later iterations could add weights by rivalry value, expected audience, travel fairness, or channel priority.

In [ ]:
def add_objective(
    model: cp_model.CpModel,
    x: AssignmentVars,
    games: Sequence[Game],
    slots: Sequence[Slot],
) -> None:
    model.Maximize(
        sum(
            x[(game.game_id, slot.slot_id)]
            for game in games
            if game.is_big_match
            for slot in slots
            if slot.is_prime_time
        )
    )

## 7. Solver Execution

We solve the model with `cp_model.CpSolver()` and set `max_time_in_seconds` to bound the search.

The solver can return several statuses. For reporting, we treat `OPTIMAL` and `FEASIBLE` as valid schedules. `INFEASIBLE` triggers diagnostic guidance about which constraints are most likely to require relaxation.

In [ ]:
def solve_model(
    model: cp_model.CpModel, max_time_in_seconds: float = 10.0
) -> Tuple[cp_model.CpSolver, int]:
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = max_time_in_seconds
    status = solver.Solve(model)
    return solver, status

## 8. Output & Visualization

The final schedule is extracted into a tidy `pandas.DataFrame` with one row per assigned game. This makes the output easy to sort, inspect, export, or feed into downstream visualization layers.

If the model is infeasible, the diagnostic block suggests relaxation candidates in a practical order: inventory, prime-time cap, team conflicts, and big-match distribution.

In [ ]:
def extract_schedule(
    solver: cp_model.CpSolver,
    status: int,
    x: AssignmentVars,
    games: Sequence[Game],
    slots: Sequence[Slot],
) -> pd.DataFrame:
    if status not in {cp_model.OPTIMAL, cp_model.FEASIBLE}:
        raise ValueError("Schedule extraction requires a feasible solver status.")

    rows = []
    slot_order = {slot.slot_id: order for order, slot in enumerate(slots)}
    for game in games:
        for slot in slots:
            if solver.BooleanValue(x[(game.game_id, slot.slot_id)]):
                rows.append(
                    {
                        "game_id": game.game_id,
                        "home_team": game.home_team,
                        "away_team": game.away_team,
                        "is_big_match": game.is_big_match,
                        "slot_id": slot.slot_id,
                        "day": slot.day,
                        "start_time": slot.start_time,
                        "channel": slot.channel,
                        "is_prime_time": slot.is_prime_time,
                        "_slot_order": slot_order[slot.slot_id],
                    }
                )

    return (
        pd.DataFrame(rows)
        .sort_values("_slot_order")
        .drop(columns="_slot_order")
        .reset_index(drop=True)
    )


def print_infeasibility_guidance() -> None:
    print("No feasible schedule found.")
    print("Consider relaxing constraints in this order:")
    print("1. Increase available slots or reduce the game set.")
    print("2. Raise max_prime_time_games if business inventory is too tight.")
    print("3. Allow controlled team conflict exceptions with penalty variables.")
    print("4. Relax big-match channel distribution for low-priority windows.")

In [ ]:
def build_model(
    teams: Sequence[Team],
    games: Sequence[Game],
    slots: Sequence[Slot],
    max_prime_time_games: int,
) -> Tuple[cp_model.CpModel, AssignmentVars]:
    model, x = initialize_model(games, slots)
    add_hard_constraints(model, x, games, slots, teams)
    add_business_constraints(model, x, games, slots, max_prime_time_games)
    add_objective(model, x, games, slots)
    return model, x


max_prime_time_games = 7
model, x = build_model(teams, games, slots, max_prime_time_games)
solver, status = solve_model(model, max_time_in_seconds=10.0)

print(f"Solver status: {solver.StatusName(status)}")
if status in {cp_model.OPTIMAL, cp_model.FEASIBLE}:
    print(f"Objective value: {solver.ObjectiveValue():.0f}")
    schedule_df = extract_schedule(solver, status, x, games, slots)
    display(schedule_df)
elif status == cp_model.INFEASIBLE:
    print_infeasibility_guidance()
else:
    print("Search ended before proving feasibility or infeasibility.")
    print("Try increasing max_time_in_seconds or simplifying the model.")